# Business question

> Which geographic areas and property types contain the highest concentration of overpriced or underpriced listings relative to their estimated market value, and how can Airbnb leverage these pricing discrepancies to optimise host revenue strategies?

# Notebook execution guidelines

A kaggle username and its PAT is needed to retrieve the data, get yours from https://www.kaggle.com/settings/api then generate a "Legacy API Credentials".
This data goes within the `.env` file.

Before running this notebook, initialize a virtual environment and install requirements:
1. `python -m venv .venv`: to create the virtual environment.
2. `.venv\scripts\activate`: to activate the virtual environment using Windows.
3. `pip install -r requirements.txt`: to install the list of requirements.

In [ ]:
import kaggle
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
import geopandas as gpd
import matplotlib.pyplot as plt
from sklearn.neighbors import BallTree
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

load_dotenv()
datasets_path = Path("dataset")

Download and read file (requires Kaggle account):

In [ ]:
if not (datasets_path / "AB_NYC_2019.csv").exists():
    kaggle.api.dataset_download_files(
        "dgomonov/new-york-city-airbnb-open-data", path=datasets_path, unzip=True
    )
df_airbnb = pd.read_csv(datasets_path / "AB_NYC_2019.csv")

In [ ]:
# City of New York: 2020 Neighborhood Tabulation Areas (NTAs) - Mapped
# https://data.cityofnewyork.us/City-Government/2020-Neighborhood-Tabulation-Areas-NTAs-Mapped/4hft-v355
url = "https://data.cityofnewyork.us/resource/9nt8-h7nd.geojson"
nyc_geojson = gpd.read_file(url)

# convert airbnb dataframe to geodataframe
gdf_airbnb = gpd.GeoDataFrame(
    df_airbnb,
    geometry=gpd.points_from_xy(df_airbnb.longitude, df_airbnb.latitude),
    crs="EPSG:4326",
)
nyc_geojson = nyc_geojson.to_crs("EPSG:4326")  # ensure same coordinate reference system
airbnb_enriched = gpd.sjoin(
    gdf_airbnb,
    nyc_geojson[["ntaname", "boroname", "geometry"]],
    how="left",
    predicate="within",
)  # assign neighborhoods and boroughs to airbnb listings

fig, ax = plt.subplots(figsize=(12, 12))
nyc_geojson.plot(
    ax=ax, color="lightgrey", edgecolor="white", linewidth=0.5
)  # draw base map of NYC neighborhoods
airbnb_enriched.plot(
    ax=ax, color="red", markersize=1, alpha=0.3
)  # Overlay airbnb listings on top of the NYC map
ax.set_title("Airbnb Listings in NYC", fontsize=14)
ax.set_axis_off()

plt.show()

In [ ]:
count_neighborhoods = airbnb_enriched["ntaname"].value_counts().reset_index()
count_neighborhoods.columns = ["ntaname", "total_listings"]

density_map = nyc_geojson.merge(
    count_neighborhoods, on="ntaname", how="left"
)  # join the counts with the GeoDataFrame to prepare for choropleth mapping

density_map["total_listings"] = density_map["total_listings"].fillna(
    0
)  # fill NaN values with 0 for neighborhoods with no listings
fig, ax = plt.subplots(figsize=(12, 12))

density_map.plot(
    column="total_listings",
    cmap="OrRd",
    linewidth=0.3,
    edgecolor="black",
    legend=True,
    legend_kwds={"shrink": 0.6, "label": "Rentals volume by NTA"},
    ax=ax,
)

ax.set_title("Airbnb market concentration by NTA", fontsize=16)
ax.set_axis_off()

plt.show()

# Data Preparation

## Data analysis
Explore how data is composed, then cleanup for ML modeling.

## Data cleaning
1. Remove null and missing values
2. Remove outliers
3. One-Hot Encoding for critical variables
4. Standardise geospatial data

In [ ]:
print("The field name of data: ", df_airbnb.columns)  # The field name of data
print("Number of fields in data: ", len(df_airbnb.columns))  # Number of fields in data
print("Number of data in data: ", len(df_airbnb))  # Number of data in data

print(df_airbnb.info)
display(df_airbnb.head(10))

In [ ]:
df_clean = airbnb_enriched.copy()
df_clean = df_clean.dropna(
    subset=["ntaname"]
)  # drop rows where 'ntaname' is NaN, which indicates listings outside of NYC boundaries
df_clean["reviews_per_month"] = df_clean["reviews_per_month"].fillna(
    0
)  # fill NaN values in 'reviews_per_month' with 0, indicating no reviews for those listings

# removing outliers
max_price = df_clean["price"].quantile(
    0.99
)  # price 0 is an error, removing the top 1% of prices to avoid skewing the model
df_clean = df_clean[(df_clean["price"] > 0) & (df_clean["price"] <= max_price)]
df_clean = df_clean[
    df_clean["minimum_nights"] <= 365
]  # minimum_nights greater than 365 are likely errors or special cases, so we filter them out

# remove unnecesary columns for ML
ml_columns = [
    "ntaname",
    "boroname",
    "latitude",
    "longitude",
    "room_type",
    "price",
    "minimum_nights",
    "number_of_reviews",
    "reviews_per_month",
    "calculated_host_listings_count",
    "availability_365",
]
df_ml = df_clean[ml_columns].copy()
df_ml["log_price"] = np.log1p(
    df_ml["price"]
)  # log1p is used to avoid issues with log(0) and to handle the skewness of the price distribution

# Export
df_ml.to_csv(datasets_path + "airbnb_nyc_ml_base.csv", index=False)
df_ml.info()

# Enhance dataset
Adding distance to public transportation, and city tourist attractions. 

In [ ]:
url_mta = "https://data.ny.gov/api/views/i9wp-a4ja/rows.csv?accessType=DOWNLOAD"
df_mta = pd.read_csv(url_mta)
col_lat_mta = (
    "Entrance Latitude" if "Entrance Latitude" in df_mta.columns else "Latitude"
)
col_lon_mta = (
    "Entrance Longitude" if "Entrance Longitude" in df_mta.columns else "Longitude"
)
df_mta = df_mta.dropna(subset=[col_lat_mta, col_lon_mta])
airbnb_coords = np.radians(
    df_ml[["latitude", "longitude"]].values
)  # convert coordiantes to radians
mta_coords = np.radians(
    df_mta[[col_lat_mta, col_lon_mta]].values
)  # convert coordiantes to radians
tree = BallTree(
    mta_coords, metric="haversine"
)  # create space tree based on subway stations
distances_rad, indices = tree.query(
    airbnb_coords, k=1
)  # Find nearest station for each airbnb listing
df_ml["dist_meters_to_subway"] = (
    distances_rad.flatten() * 6371000
)  # Convert radians to meters

# Check
print(df_ml[["latitude", "longitude", "dist_meters_to_subway"]].head())

In [ ]:
# adding points of interest: we considered "https://data.cityofnewyork.us/City-Government/CommonPlace/rxuy-2muj" but has too many points
# we are going with strategic points of interest that are relevant to tourists and visitors of NYC
pois = [
    ("Times Square", 40.7580, -73.9855),
    ("Central Park", 40.7644, -73.9730),
    ("Met Museum", 40.7794, -73.9632),
    ("Empire State", 40.7484, -73.9857),
    ("Washington Square", 40.7308, -73.9973),
    ("SoHo", 40.7233, -73.9988),
    ("Financial District", 40.7074, -74.0113),
    ("Penn Station", 40.7505, -73.9934),
    ("Grand Central", 40.7527, -73.9772),
    ("Williamsburg", 40.7160, -73.9587),
    ("Barclays Center", 40.6826, -73.9754),
    ("Long Island City", 40.7465, -73.9455),
]
poi_coords = np.radians([[lat, lon] for _, lat, lon in pois])
airbnb_coords = np.radians(df_ml[["latitude", "longitude"]].values)
tree_poi = BallTree(poi_coords, metric="haversine")
dist_rad, _ = tree_poi.query(airbnb_coords, k=1)
df_ml["dist_min_poi_meters"] = dist_rad.flatten() * 6371000

print(df_ml[["latitude", "longitude", "dist_min_poi_meters"]].head())

In [ ]:
# Final encoding for ML - categorical variables to numbers
# we discard ntaname due high cardinality, we keep boroname as a proxy for aggregated location
df_final = pd.get_dummies(
    df_ml.drop(columns=["ntaname"]), columns=["room_type", "boroname"], drop_first=True
)

X = df_final.drop(columns=["price", "log_price"])
y = df_final["log_price"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=11
)

# train baseline (random forest)
model = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=11)
model.fit(X_train, y_train)

# Initial evaluation
preds = model.predict(X_test)
print(f"R2 Score over base model: {r2_score(y_test, preds):.4f}")

# Analysis of variable importance to justify which variables are effective
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(
    ascending=False
)
print("\nVariable importance:")
print(importances)

# Export
df_final.to_csv(datasets_path + "airbnb_nyc_ml_final.csv", index=False)

In [ ]:
# calculate predictions and anomalies (using expm1 because we trained with log1p)
df_ml["predicted_price"] = np.expm1(model.predict(X))
df_ml["residue_error"] = df_ml["price"] - df_ml["predicted_price"]

# identify anomalies: overpriced if real price > 50% of predicted, underpriced if real price < 50% of predicted
df_ml["status"] = "normal"
df_ml.loc[df_ml["residue_error"] > (df_ml["predicted_price"] * 0.5), "status"] = (
    "overpriced"
)
df_ml.loc[df_ml["residue_error"] < -(df_ml["predicted_price"] * 0.5), "status"] = (
    "underpriced"
)

# visual importance of business opportunity
fig, ax = plt.subplots(figsize=(12, 12))
nyc_geojson.plot(ax=ax, color="lightgrey", edgecolor="white", linewidth=0.5)

anomalies = df_ml[df_ml["status"] != "normal"]
colors = {"overpriced": "red", "underpriced": "green"}

for status, color in colors.items():
    subset = anomalies[anomalies["status"] == status]
    ax.scatter(
        subset["longitude"], subset["latitude"], c=color, s=8, label=status, alpha=0.4
    )

ax.set_title("Market anomalies over NTAs", fontsize=14)
ax.set_axis_off()
ax.legend(loc="upper right", markerscale=2)

plt.show()